# **NLP TEXT CLASSIFIER**

Step 1 : Install and import

In [ ]:
!pip install transformers datasets torch

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer

Step 2 : Load dataset

In [ ]:
dataset = load_dataset("Tobi-Bueck/customer-support-tickets")

print(dataset)
print(dataset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'version', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8'],
        num_rows: 61765
    })
})
{'subject': 'Wesentlicher Sicherheitsvorfall', 'body': 'Sehr geehrtes Support-Team,\\n\\nich möchte einen gravierenden Sicherheitsvorfall melden, der gegenwärtig mehrere Komponenten unserer Infrastruktur betrifft. Betroffene Geräte umfassen Projektoren, Bildschirme und Speicherlösungen auf Cloud-Plattformen. Der Grund für die Annahme ist, dass der Vorfall eine potenzielle Datenverletzung im Zusammenhang mit einer Cyberattacke darstellt, was ein erhebliches Risiko für sensible Informationen und den laufenden Geschäftsbetrieb unserer Organisation bedeutet.\\n\\nUnsere initialen Untersuchungen haben ungewöhnliche Aktivitäten und Abweichungen bei den Geräten ergeben. Trotz der Umsetzung unserer standardisierten Behebungs- und Eindämmungsmaßnahmen konnt

Step 3 : Understand Labels

In [ ]:
print(dataset['train'].column_names)

['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'version', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8']


In [ ]:
labels = list(set(dataset['train']['type']))
print("Labels:", labels)

Labels: [None, 'Problem', 'Incident', 'Request', 'Change']


Step 4 : Tokenization

In [ ]:
dataset = dataset.filter(lambda x: x['type'] is not None)

Filter:   0%|          | 0/61765 [00:00<?, ? examples/s]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
def tokenize(example):
    text = [
        (str(s) if s is not None else "") + " " + (str(b) if b is not None else "")
        for s, b in zip(example['subject'], example['body'])
    ]
    return tokenizer(text, padding="max_length", truncation=True)

In [ ]:
dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/48587 [00:00<?, ? examples/s]

Step 5 : Convert Labels to Numbers

In [ ]:
labels = list(set(dataset['train']['type']))

label_map = {label: i for i, label in enumerate(labels)}

def encode_labels(example):
    example['labels'] = label_map[example['type']]
    return example

dataset = dataset.map(encode_labels)

Map:   0%|          | 0/48587 [00:00<?, ? examples/s]

Step 6 : Format Dataset

In [ ]:
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Step 7 : Load Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_map)
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step 8 : Train Model

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    num_train_epochs=1,
    logging_steps=10
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"].select(range(300))
)

trainer.train()

Step,Training Loss
10,0.892947
20,0.865346
30,0.712591


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=38, training_loss=0.7948271098889803, metrics={'train_runtime': 990.0257, 'train_samples_per_second': 0.303, 'train_steps_per_second': 0.038, 'total_flos': 39741637017600.0, 'train_loss': 0.7948271098889803, 'epoch': 1.0})

Step 9 : Evaluation

In [ ]:
trainer.evaluate(dataset["train"].select(range(100)))

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.6198769211769104,
 'eval_runtime': 85.0419,
 'eval_samples_per_second': 1.176,
 'eval_steps_per_second': 0.153,
 'epoch': 1.0}

Step 10 : Prediction

In [ ]:
text = "My system is not working"

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

outputs = model(**inputs)

pred = torch.argmax(outputs.logits, dim=1).item()

print("Prediction:", labels[pred])

Prediction: Incident
